# Customer Churn Prediction - Telco Dataset

In [ ]:
# 📌 Customer Churn Prediction - Telco Dataset

# ==========================
# 1. Import Libraries
# ==========================

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

# ==========================
# 2. Create Images Folder
# ==========================

os.makedirs("images", exist_ok=True)

# ==========================
# 3. Load Dataset
# ==========================

try:
    df = pd.read_csv("churn_data.csv")
except FileNotFoundError:
    raise FileNotFoundError(
        "Could not find 'churn_data.csv'.\n"
        "Download the IBM Telco Customer Churn dataset and place it "
        "in the same folder as this notebook."
    )

print("Dataset Loaded Successfully\n")
print(df.head())

# ==========================
# 4. Dataset Information
# ==========================

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Info:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

# ==========================
# 5. Data Cleaning
# ==========================

if "customerID" in df.columns:
    df.drop("customerID", axis=1, inplace=True)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].median()
)

# ==========================
# 6. Encode Categorical Data
# ==========================

encoder = LabelEncoder()

for column in df.columns:
    if df[column].dtype == object and column != "Churn":
        df[column] = encoder.fit_transform(df[column])

df["Churn"] = df["Churn"].map(
    {
        "Yes": 1,
        "No": 0,
    }
)

# ==========================
# 7. Visualization
# ==========================

plt.figure(figsize=(6, 4))

sns.countplot(
    x="Churn",
    data=df
)

plt.title("Customer Churn Distribution")
plt.tight_layout()

plt.savefig(
    "images/churn_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ==========================
# 8. Feature Scaling
# ==========================

scaler = StandardScaler()

numerical_columns = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]

df[numerical_columns] = scaler.fit_transform(
    df[numerical_columns]
)

# ==========================
# 9. Train/Test Split
# ==========================

X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

# ==========================
# 10. Logistic Regression
# ==========================

lr_model = LogisticRegression(
    random_state=42
)

lr_model.fit(
    X_train,
    y_train,
)

lr_predictions = lr_model.predict(X_test)

print("\n==============================")
print("Logistic Regression Results")
print("==============================")

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        lr_predictions,
    ),
)

print(
    classification_report(
        y_test,
        lr_predictions,
    )
)

# ==========================
# 11. Random Forest
# ==========================

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
)

rf_model.fit(
    X_train,
    y_train,
)

rf_predictions = rf_model.predict(
    X_test
)

print("\n==============================")
print("Random Forest Results")
print("==============================")

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        rf_predictions,
    ),
)

print(
    classification_report(
        y_test,
        rf_predictions,
    )
)

# ==========================
# 12. Confusion Matrix
# ==========================

cm = confusion_matrix(
    y_test,
    rf_predictions,
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
)

plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()

plt.savefig(
    "images/confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nImages saved successfully!")

print("images/churn_distribution.png")
print("images/confusion_matrix.png")